# Spatio-temporal prediction — Swiss temperature data

Demonstration notebook for the temperature experiment from:
> Amato et al. (2020) *A Novel Framework for Spatio-Temporal Prediction of Environmental Data Using Deep Learning*, Scientific Reports.

**Prerequisite:** Place the temperature data files under `data/temperature/` (see README). The data is not included in the repository.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from stdl.config import ExperimentConfig
from stdl.data import load_data
from stdl.decomposition import svd_decompose
from stdl.evaluate import control_stats, spatial_mae, temporal_mae
from stdl.model import build_model
from stdl.train import train

## 1. Load config and data

In [ ]:
cfg = ExperimentConfig.from_yaml(Path("../configs/temperature.yaml"))
bundle = load_data(cfg.data)
print(f"Train stations: {bundle.X_train.shape[0]}, Val: {bundle.X_val.shape[0]}, Test: {bundle.X_test.shape[0]}")
print(f"Prediction mesh points: {bundle.X_mesh.shape[0]}")
print(f"Timesteps: {bundle.y_train.shape[1]}")

## 2. SVD decomposition (EOF analysis)

Scree plot — dashed line marks the selected number of EOFs (capturing ~95% of variance).

In [ ]:
import tensorflow as tf

svd = svd_decompose(bundle.y_svd, cfg.model.n_eofs)

Z = tf.cast(bundle.y_svd, tf.float32)
nS = tf.constant(Z.shape[0], dtype=tf.float32)
time_mean_v = tf.reduce_mean(Z, axis=0)
Ztilde = (Z - time_mean_v) / tf.sqrt(nS - 1)
s_full, _, _ = tf.linalg.svd(Ztilde)
cum_var = np.cumsum(np.array(s_full) ** 2) / np.sum(np.array(s_full) ** 2)

fig, ax = plt.subplots(figsize=(6, 2.5))
ax.plot(np.arange(1, 51), cum_var[:50], c="black")
ax.axvline(cfg.model.n_eofs, color="black", linestyle="--", linewidth=1)
ax.set_ylabel("Cumulative\nexplained variance")
ax.set_xlabel("Number of components")
plt.tight_layout()
plt.show()

## 3. Inspect first three EOF components

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 8))
timestamps = bundle.y_train.shape[1]
tick_positions = [0, timestamps // 2, timestamps - 1]

for k in range(3):
    axes[0, k].plot(np.array(svd.v[:, k]), color="black", alpha=0.5)
    axes[0, k].set_title(f"Temporal basis (EOF {k+1})")

    sc = axes[1, k].scatter(
        bundle.raw_coords_train[:, 0],
        bundle.raw_coords_train[:, 1],
        c=np.array(svd.u[:, k]),
    )
    plt.colorbar(sc, ax=axes[1, k])
    axes[1, k].set_title(f"Spatial coefficients (EOF {k+1})")
    axes[1, k].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 4. Build and train model

In [ ]:
model = build_model(cfg.model, svd)
model.summary()

In [ ]:
history = train(model, bundle, svd, cfg.train)

## 5. Predictions and evaluation

In [ ]:
y_hat_test, y_coef_test = model.predict(bundle.X_test, verbose=0)
y_hat_mesh, y_coef_mesh = model.predict(bundle.X_mesh, verbose=0)

mae_t = temporal_mae(bundle.y_test, y_hat_test)
mae_s = spatial_mae(bundle.y_test, y_hat_test)
print(f"Mean temporal MAE : {mae_t.mean():.4f} °C")
print(f"Mean spatial MAE  : {mae_s.mean():.4f} °C")

In [ ]:
fig, ax = plt.subplots(figsize=(18, 4))
ax.plot(mae_t, c="black", alpha=0.5, linewidth=0.8, label="Temporal MAE")
ax.axhline(mae_t.mean(), c="red", linestyle="--", alpha=0.7, label=f"Mean ({mae_t.mean():.3f} °C)")
ax.set_xlabel("Timestep")
ax.set_ylabel("MAE (°C)")
ax.set_title("Test MAE over time")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Random test station and timestamp
rng = np.random.default_rng(0)
loc = rng.integers(0, bundle.X_test.shape[0])
timestamp = rng.integers(0, bundle.y_test.shape[1])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (vals, title) in zip(
    axes,
    [(bundle.y_test[:, timestamp], "True (test stations)"),
     (y_hat_test[:, timestamp], "Predicted (test stations)")],
):
    sc = ax.scatter(
        bundle.raw_coords_test[:, 0],
        bundle.raw_coords_test[:, 1],
        c=vals, vmin=0, vmax=28,
    )
    plt.colorbar(sc, ax=ax, label="Temperature (°C)")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Mesh prediction maps
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (vals, title) in zip(
    axes,
    [(y_hat_mesh[:, timestamp], f"Mesh prediction (t={timestamp}, {cfg.model.n_eofs} EOFs)"),
     (y_coef_mesh[:, 0], "Modelled spatial coefficients (EOF 1)")],
):
    sc = ax.scatter(
        bundle.raw_coords_mesh[:, 0],
        bundle.raw_coords_mesh[:, 1],
        c=vals, marker=",", s=1.5,
    )
    plt.colorbar(sc, ax=ax)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 6. Control statistics

In [ ]:
stats = control_stats(bundle.y_train, y_hat_mesh)

fig, axes = plt.subplots(2, 1, figsize=(13, 8))
for ax, col, title in zip(
    axes,
    ["y_max", "y_min"],
    ["Maximum values", "Minimum values"],
):
    ax.plot(stats[col].values, label="Training data", color="black", linewidth=1.2)
    ax.plot(stats[col.replace("y_", "mesh_")].values, label="Predicted (mesh)", color="orange", linewidth=0.9)
    ax.set_title(f"Train vs predicted {title}")
    ax.set_ylabel("Temperature (°C)")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()